# 04 Test: the chosen model on the test period, once

**Purpose:** Evaluate the final model, chosen on the backtest only, on the test period (2024-10 to 2026-09), together with the best simple baseline. This is the only notebook that uses the test period.
**Input:** `../data/processed/asylum_clean.csv` (from `01_preprocess.ipynb`) and `../output/tables/03_backtest_errors.csv` (from `03_evaluate.ipynb`).
**Output:** `../data/processed/test_forecasts.csv`, `../output/tables/04_test_errors.csv` and `../output/figures/04_test_forecasts_h2.png`.

**The choice was made before this notebook was run** (notebook 209, Step 7): `03_evaluate.ipynb`, section 8, chooses the **linear regression** of `02b` (it is as good as Ridge at $h = 2$ and simpler). That section was saved on GitHub before the test (commit `c6a3976`, 17 November 2026). We do not change the choice after seeing the results below; they are reported as they are (notebook 209, Step 8).

**Timeline.** On 23 November 2026 we added one more simple rule, the **random walk** (`02_estimate.ipynb`), to this notebook. It is reported for information only: it is not a candidate model, and the chosen model, its forecasts and its test result are exactly the same as in the first run (commit `4f284f5`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

np.random.seed(42)
plt.style.use("seaborn-v0_8-muted")

## 1. Load the whole series

For the first time, we use all 225 months, including the 24 months of the test period.

In [ ]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")

y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)
months = clean.index

last_backtest = int(np.where(months == pd.Timestamp("2024-09-01"))[0][0])
last_month = len(months) - 1
H_MAX = 3

print(clean["period"].value_counts())
print("Last backtest month:", months[last_backtest], "| last month:", months[last_month])

## 2. The models, exactly as in the backtest

- **Linear regression** (`02b_estimate_regression.ipynb`): one model per horizon, predicting the change of log applications from the last two monthly changes, the same change one year earlier and the calendar month. The function `make_table` is copied without any change.
- **12-month average** (`02_estimate.ipynb`): the average of the last 12 known months, for every horizon.
- **Random walk** (`02_estimate.ipynb`, added on 23 November): the last known month, for every horizon.

In [ ]:
def make_table(h):
    # Copied from 02b: one row per origin t, features known at t, target = change of log from t to t + h
    rows = []
    for t in range(12, len(log_y) - h):
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    X = pd.concat([table[["change_last_month", "change_month_before", "same_change_last_year"]], dummies], axis=1)
    return X, table["target"]


def random_walk(train_y, h_max):
    # Copied from 02: the last known month, the same for every horizon
    return np.log(np.full(h_max, train_y[-1]))


def average_12(train_y, h_max):
    # Copied from 02: average of the last 12 known months, the same for every horizon
    average = train_y[-12:].mean()
    return np.log(np.full(h_max, average))


tables = {h: make_table(h) for h in range(1, H_MAX + 1)}

## 3. Forecasts for the test period

We continue the backtest into the test period with the same rule: at each origin, the models are fitted on **all months known at that origin and nothing after it**, and we forecast $h = 1, 2, 3$. The origins go from September 2024 (the last backtest month) to August 2026. So the forecasts for, say, March 2026 are made with the months up to December 2025 ($h = 3$), January 2026 ($h = 2$) or February 2026 ($h = 1$), as a planner would have had them.

This is the time-series version of "refit on all training data, then test" (notebook 209, Step 8): there is no single refit, because in practice the model is refitted every month when a new number arrives. We expect 24 forecasts for $h = 1$, 23 for $h = 2$ and 22 for $h = 3$ per model.

In [ ]:
rows = []

for t in range(last_backtest, last_month):          # origins 2024-09 to 2026-08
    for h in range(1, H_MAX + 1):
        target_pos = t + h
        if target_pos > last_month:
            continue

        # Linear regression: training rows whose target is known at the origin
        X, target = tables[h]
        train_rows = list(range(X.index.min(), t - h + 1))
        assert max(train_rows) + h <= t                     # no training target after the origin
        model = make_pipeline(StandardScaler(), LinearRegression()).fit(X.loc[train_rows], target.loc[train_rows])
        forecast_lr = log_y[t] + model.predict(X.loc[[t]])[0]

        # 12-month average: the last 12 months up to the origin
        forecast_avg = average_12(y[: t + 1], H_MAX)[h - 1]

        # Random walk: the last month up to the origin (added on 23 November, for information)
        forecast_rw = random_walk(y[: t + 1], H_MAX)[h - 1]

        for name, forecast_log in [("Linear regression", forecast_lr), ("12-month average", forecast_avg),
                                   ("Random walk", forecast_rw)]:
            rows.append({
                "model": name, "origin": months[t], "month": months[target_pos], "h": h,
                "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                "forecast": np.exp(forecast_log), "actual": y[target_pos],
            })

test_forecasts = pd.DataFrame(rows)
print(test_forecasts.groupby(["model", "h"]).size())
print()
print("All forecast months in the test period:", (test_forecasts["month"] >= pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (test_forecasts["month"] > test_forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(test_forecasts["forecast_log"])).sum())

**What we see.** Each model has 24 forecasts for $h = 1$, 23 for $h = 2$ and 22 for $h = 3$, as expected. All forecast months lie in the test period and after their origin, and no forecast is missing. Together with the `assert` in the loop, this shows that each forecast used only the months known at its origin.

## 4. Test errors

The same two error measures as in `03_evaluate.ipynb`: the MAE in persons (main) and the RMSE in logs.

In [ ]:
test_forecasts["abs_error"] = (test_forecasts["actual"] - test_forecasts["forecast"]).abs()
test_forecasts["sq_error_log"] = (test_forecasts["actual_log"] - test_forecasts["forecast_log"]) ** 2

grouped = test_forecasts.groupby(["model", "h"])
test_errors = pd.DataFrame({
    "MAE (persons)": grouped["abs_error"].mean(),
    "RMSE (log)": np.sqrt(grouped["sq_error_log"].mean()),
    "forecasts": grouped["abs_error"].size(),
}).reset_index()

test_errors.to_csv("../output/tables/04_test_errors.csv", index=False)
test_errors.round(3)

### The main comparison at $h = 2$

In [ ]:
h2 = test_errors[test_errors["h"] == 2].set_index("model")
mae_lr, mae_avg = h2.loc["Linear regression", "MAE (persons)"], h2.loc["12-month average", "MAE (persons)"]
rmse_lr, rmse_avg = h2.loc["Linear regression", "RMSE (log)"], h2.loc["12-month average", "RMSE (log)"]

print(f"Test MAE at h=2: linear regression {mae_lr:.1f} persons, 12-month average {mae_avg:.1f} persons")
print(f"Improvement: {100 * (1 - mae_lr / mae_avg):.0f}% lower MAE, {100 * (1 - rmse_lr / rmse_avg):.0f}% lower RMSE (log)")
print("Success rule of our plan (at least 10% lower MAE than the best simple baseline):", mae_lr <= 0.9 * mae_avg)
print(f"For comparison, average actual number of applications in these months: {test_forecasts[test_forecasts['h'] == 2]['actual'].mean():.0f}")

mae_rw, rmse_rw = h2.loc["Random walk", "MAE (persons)"], h2.loc["Random walk", "RMSE (log)"]
print()
print(f"For information (added on 23 November): random walk {mae_rw:.1f} persons; "
      f"linear regression {100 * (1 - mae_lr / mae_rw):.0f}% lower MAE, {100 * (1 - rmse_lr / rmse_rw):.0f}% lower RMSE (log)")

**What we see.**
- At $h = 2$, the linear regression misses by **188 applications per month** on average, against 331 for the 12-month average: **43% lower MAE** and 41% lower RMSE (log) (0.112 against 0.191). Our success rule (at least 10% better than the best simple baseline) is met.
- In these months there were on average 1,982 applications per month, so the model misses by about a tenth of a typical month.
- The ranking is the same at the other horizons: 149 against 297 at $h = 1$, 217 against 359 at $h = 3$ (table above).
- **For information, the random walk** (added on 23 November) misses by 269 at $h = 2$: our model is 30% lower by MAE and by RMSE (log). At $h = 1$ the random walk is much closer (187 against 149), at $h = 3$ it is as bad as the 12-month average (358).

**What this means for us.** The model we chose on the backtest also clearly beats the simple rule on months it has never seen. The gain is in the same direction as in the backtest (32% there), so the result of the backtest holds up on new data.

## 5. Test against backtest

How do the test errors compare with the backtest errors of the same models (from `03_evaluate.ipynb`)? The backtest error of the chosen model is a little too optimistic, because it won the comparison (notebook 108b). The two periods also differ in how hard they are: we put the average number of applications next to the errors.

In [ ]:
backtest = pd.read_csv("../output/tables/03_backtest_errors.csv").set_index("model")

comparison = pd.DataFrame(index=["Linear regression", "12-month average", "Random walk"])
for h in [1, 2, 3]:
    comparison[f"backtest MAE h={h}"] = backtest.loc[comparison.index, f"MAE h={h}"]
    comparison[f"test MAE h={h}"] = [test_errors[(test_errors["model"] == m) & (test_errors["h"] == h)]["MAE (persons)"].iloc[0]
                                     for m in comparison.index]
print(comparison.round(0))
print()
level = clean.groupby("period")["applicants_total"].agg(["mean", "std"]).round(0)
print("Applications per month in each period:")
print(level)

**What we see.**
- All three have **much smaller errors in the test period** than in the backtest: at $h = 2$ the linear regression 188 instead of 358, the 12-month average 331 instead of 528, the random walk 269 instead of 379.
- The test period is much calmer: the standard deviation of the monthly applications is 338, against 888 in the backtest, for a similar average (2,008 against 1,852).

**What this means for us.** The smaller test errors come mostly from an **easier period**, not from a model that got better: the simple rules improve too. The gap to the random walk is larger in the test (30%) than in the backtest (5%); with only 23 forecasts we read this as a hint, not as proof. This is why we compare the two models on the same months, and not the test error with the backtest error. With only 23 forecasts at $h = 2$, the test MAE is also uncertain; we report it as one honest check, not as the error to expect in every future year. In a new crisis like 2015-16, the errors would be much larger (`03_evaluate.ipynb`, section 5).

## 6. The forecasts two months ahead, month by month

The table shows what a planner would have seen: for each month of the test period, the forecast made two months earlier and the actual number.

In [ ]:
h2_forecasts = test_forecasts[test_forecasts["h"] == 2]
monthly = h2_forecasts.pivot(index="month", columns="model", values="forecast").round(0)
monthly.insert(0, "Actual", h2_forecasts.groupby("month")["actual"].first())
monthly["error of the linear regression"] = monthly["Linear regression"] - monthly["Actual"]
monthly

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(monthly.index, monthly["Actual"], color="black", linewidth=2, marker="o", label="Actual")
ax.plot(monthly.index, monthly["Linear regression"], color=plt.cm.tab10(6), marker="o", label="Linear regression (forecast 2 months ahead)")
ax.plot(monthly.index, monthly["12-month average"], color=plt.cm.tab10(1), marker="o", label="12-month average (forecast 2 months ahead)")
ax.plot(monthly.index, monthly["Random walk"], color=plt.cm.tab10(2), linestyle="--", marker=".", label="Random walk (added on 23 November)")
ax.set_title("Test period: forecasts two months ahead (h = 2), 2024-11 to 2026-09")
ax.set_xlabel("Forecast month")
ax.set_ylabel("Applications per month")
ax.set_ylim(bottom=0)
ax.legend(loc="lower left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../output/figures/04_test_forecasts_h2.png")
plt.show()

**What we see.**
- The 12-month average is almost a flat line around 2,000 to 2,500: it cannot follow the rise in summer and autumn 2025 or the fall in early 2026.
- The linear regression follows the seasonal pattern (the rise from June to October 2025 and again from June 2026). In 14 of the 23 months, it misses by 200 or less (table above).
- Its largest errors come right **after turning points**: it was too low in March to May 2025 (−183 to −317) and too high in November 2025 to February 2026 (+470 in November, +364 in January), when the numbers had already turned down.
- The random walk (dashed) is the actual curve shifted by two months: it is late at every turn, for example 2,645 for December 2025 when the actual number was 2,015.

**What this means for us.** The model reacts to the newest changes, but only once they are visible in the data. A planner can use it as a monthly guide, but should expect misses of a few hundred applications at turning points.

## 7. Save the test forecasts

In [ ]:
test_forecasts.drop(columns=["abs_error", "sq_error_log"]).to_csv("../data/processed/test_forecasts.csv", index=False)
print("Saved", test_forecasts.shape[0], "test forecasts")

## Summary

- **Choice (made before this notebook, `03_evaluate.ipynb` section 8):** the linear regression on the series' own past.
- **Test result at $h = 2$:** MAE 188 applications per month against 331 for the 12-month average, **43% lower**; RMSE (log) 41% lower. The success rule of our plan is met, and the ranking holds at $h = 1$ and $h = 3$. For information, the random walk (added on 23 November) is at 269: our model is 30% lower.
- **Careful reading:** the test period (2024-10 to 2026-09) was calm, so all three have smaller errors than in the backtest; only 23 forecasts at $h = 2$; the largest misses come after turning points.
- **The test period was used once.** The choice was not changed after seeing these results; the random walk, added afterwards, changes neither the choice nor the result of the chosen model.